# D-FINE motifs_v1 · PACE-ICE chained 8-hour LLM-GE commands

Copy these Bash blocks into a PACE terminal in order. Start with a fresh tag after a failed generation-zero run. Section 5 queues automatic continuation windows with the first controller.


## 1. Start a fresh experiment: configure and generate scripts

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
module load uv
git pull --ff-only origin fred-custom-seeds
uv sync
export FRED_SEED_FAMILY=dfine FRED_DFINE_SEARCH_SPACE=motifs_v1
export FRED_DFINE_MAX_INFERENCE_MS_PER_FRAME=10
export FRED_DFINE_MAX_PEAK_MEMORY_BYTES=3000000000
export FRED_NUM_GENERATIONS=21
export FRED_RUN_TAG="dfine_$(date +%Y%m%d)_eot0_trial$(date +%H%M%S)"
export LLMGE_PORT=8479 LLMGE_CLUSTER=pace-ice
export LOCAL=false LOCAL_LLM=true INFERENCE_SUBMISSION=false
export LLMGE_LLM_MODEL=mixtral SUBMIT_ISLAND_CONTROLLER=0
export HF_HUB_DISABLE_XET=1
unset FRED_EVENTS LLMGE_BOUNDED_SESSION LLMGE_MODEL_PATH LLM_MAX_NEW_TOKENS
mkdir -p run_job_outputs/server run_job_outputs/islands
printf '%s
' "$FRED_RUN_TAG" > run_job_outputs/dfine_current_tag.txt
printf 'tag=%s commit=%s
' "$FRED_RUN_TAG" "$(git rev-parse --short HEAD)"
if test ! -e "fred_runs/$FRED_RUN_TAG" \
   && test -x .venv/bin/python \
   && test -s sota/FRED_LLM_GE/phase1_detection/seeds/yolo11/data_cache/challenging/search_train_step50/data.yaml \
   && test -s sota/FRED_LLM_GE/phase1_detection/seeds/yolo11/data_cache/challenging/search_val_step10/data.yaml; then
  echo 'Fresh tag, Python environment, and search data ready'
  .venv/bin/python slurm.py
  printf '%s\n' "$FRED_RUN_TAG" > run_job_outputs/dfine_current_tag.txt
else
  echo 'Preflight failed. Do not submit jobs.'
fi
```

## 2. Submit the 8-hour LLM server

```bash
if SERVER_JOB_ID=$(sbatch --parsable --partition=coe-gpu --export=ALL server.sh); then
  export LLMGE_SERVER_JOB_ID=${SERVER_JOB_ID%%;*}
  printf '%s
' "$LLMGE_SERVER_JOB_ID" > run_job_outputs/dfine_current_server.id
  echo "Server job: $LLMGE_SERVER_JOB_ID"
  squeue -j "$LLMGE_SERVER_JOB_ID" -o '%i %T %R %L'
else
  echo 'Server submission failed. Do not start the controller.'
fi
```

## 3. Check server readiness after it starts running

```bash
tail -F "run_job_outputs/server/slurm-$LLMGE_SERVER_JOB_ID.out"
```

Wait for `Uvicorn running`, `Serving on port 8479`, and `Skipping island controller submission`. Press Ctrl-C to leave `tail`.

## 4. Smoke check all three required motif operators

```bash
export SMOKE_ROOT="$PWD/fred_runs/$FRED_RUN_TAG/operator_smoke"
mkdir -p "$SMOKE_ROOT"
if SMOKE_JOB_ID=$(sbatch --parsable --export=ALL --array=0-2%3 \
  -c 2 --mem=16G -t 0:30:00 --job-name=dfine-motif-smoke \
  --output="$SMOKE_ROOT/slurm-%A_%a.out" \
  --wrap='cd "$SLURM_SUBMIT_DIR"; module load uv; case "$SLURM_ARRAY_TASK_ID" in 0) target=backbone;; 1) target=encoder;; 2) target=decoder;; esac; .venv/bin/python src/llm_mutation.py sota/FRED_LLM_GE/seeds/seed_dfine.py "$SMOKE_ROOT/network_${target}.py" templates/FRED_DFINE/MotifsV1/1.txt --llm_model mixtral --coverage_target "$target" --dfine_search_space motifs_v1'); then
  SMOKE_JOB_ID=${SMOKE_JOB_ID%%;*}
  echo "Smoke array: $SMOKE_JOB_ID"
else
  echo 'Smoke submission failed. Do not start the controller.'
fi
```

After all three tasks leave `squeue`:

```bash
sacct -j "$SMOKE_JOB_ID" --format=JobID,State,ExitCode -P
for target in backbone encoder decoder; do
  test -s "$SMOKE_ROOT/network_${target}.py" && test -s "$SMOKE_ROOT/network_${target}.py.genome.json" && echo "$target ready"
done
```

Continue only when all three array tasks show `COMPLETED|0:0` and all three targets print `ready`.


## 5. Submit one controller and queue automatic 8-hour continuations

Choose how many additional server windows to queue after the first. Three extra windows allow up to four 8-hour server allocations. The next window starts only after the previous controller exits successfully, its server allocation ends, and `status.json` records a normal `paused_for_server_refresh`. A completed run stops the chain. A failed controller blocks it for investigation.

```bash
export FRED_CHAIN_EXTRA_WINDOWS=3
export FRED_CHAIN_ROOT="$PWD/fred_runs/$FRED_RUN_TAG/chain"
mkdir -p "$FRED_CHAIN_ROOT"
if RUN_JOB_ID=$(sbatch --parsable --export=ALL run.sh); then
  RUN_JOB_ID=${RUN_JOB_ID%%;*}
  printf '%s\n' "$RUN_JOB_ID" > run_job_outputs/dfine_current_controller.id
  printf 'server\t%s\tfirst\ncontroller\t%s\t%s\n' \
    "$LLMGE_SERVER_JOB_ID" "$RUN_JOB_ID" "$LLMGE_SERVER_JOB_ID" > "$FRED_CHAIN_ROOT/jobs.tsv"
  echo "Controller job: $RUN_JOB_ID"
  if [[ $FRED_CHAIN_EXTRA_WINDOWS =~ ^[1-9][0-9]*$ ]] && \
     CHAIN_JOB_ID=$(sbatch --parsable \
       --dependency="afterok:${RUN_JOB_ID},afterany:${LLMGE_SERVER_JOB_ID}" \
       --export=ALL --output="$FRED_CHAIN_ROOT/transition-%j.out" \
       dfine_chain_continue.sbatch "$RUN_JOB_ID" "$LLMGE_SERVER_JOB_ID" \
       "$FRED_CHAIN_EXTRA_WINDOWS"); then
    CHAIN_JOB_ID=${CHAIN_JOB_ID%%;*}
    printf 'transition\t%s\t%s\n' "$CHAIN_JOB_ID" "$RUN_JOB_ID" >> "$FRED_CHAIN_ROOT/jobs.tsv"
    echo "Automatic continuation queued: $CHAIN_JOB_ID"
    squeue -j "$RUN_JOB_ID,$CHAIN_JOB_ID" -o '%i %T %j %R %L'
  else
    echo 'Controller submitted, but automatic continuation was not queued. Investigate before the server window ends.'
  fi
else
  echo 'Controller submission failed. No continuation was queued.'
fi
```


## 6. Monitor the active run

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
export FRED_RUN_TAG="$(cat run_job_outputs/dfine_current_tag.txt)"
RUN_JOB_ID="$(cat run_job_outputs/dfine_current_controller.id)"
export FRED_CHAIN_ROOT="$PWD/fred_runs/$FRED_RUN_TAG/chain"
test -f "$FRED_CHAIN_ROOT/jobs.tsv" && cat "$FRED_CHAIN_ROOT/jobs.tsv"
squeue -u "$USER" -o '%i %T %j %R %L'
sacct -j "$RUN_JOB_ID" --format=JobID,State,ExitCode -P
test -f "fred_runs/$FRED_RUN_TAG/status.json" && cat "fred_runs/$FRED_RUN_TAG/status.json"
tail -n 80 "run_job_outputs/islands/slurm-$RUN_JOB_ID.out"
```

## 7. Resume after the server window ends

Use this manual fallback only when no automatic transition is pending. Run only after the previous controller has exited and the saved status indicates a normal pause. Reuse the **same** tag, code, generation count, limits, and generated scripts.

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
module load uv
export FRED_SEED_FAMILY=dfine FRED_DFINE_SEARCH_SPACE=motifs_v1
export FRED_DFINE_MAX_INFERENCE_MS_PER_FRAME=10
export FRED_DFINE_MAX_PEAK_MEMORY_BYTES=3000000000
export FRED_NUM_GENERATIONS=21
export FRED_RUN_TAG="$(cat run_job_outputs/dfine_current_tag.txt)"
export LLMGE_PORT=8479 LLMGE_CLUSTER=pace-ice
export LOCAL=false LOCAL_LLM=true INFERENCE_SUBMISSION=false
export LLMGE_LLM_MODEL=mixtral SUBMIT_ISLAND_CONTROLLER=0
export HF_HUB_DISABLE_XET=1
unset FRED_EVENTS LLMGE_BOUNDED_SESSION LLMGE_MODEL_PATH LLM_MAX_NEW_TOKENS
PREV_RUN_JOB_ID="$(cat run_job_outputs/dfine_current_controller.id)"
sacct -j "$PREV_RUN_JOB_ID" --format=JobID,State,ExitCode -P
cat "fred_runs/$FRED_RUN_TAG/status.json"
if ! ACTIVE_STATE=$(squeue -h -j "$PREV_RUN_JOB_ID" -o '%T'); then
  echo 'Could not verify the previous controller. Do not resume yet.'
elif test -n "$ACTIVE_STATE"; then
  echo "Previous controller is still $ACTIVE_STATE. Do not resume yet."
elif ! .venv/bin/python -c 'import json,sys; s=json.load(open(sys.argv[1])); assert s.get("status") == "paused_for_server_refresh", s' "fred_runs/$FRED_RUN_TAG/status.json"; then
  echo 'Run is not paused for server refresh. Investigate before resuming.'
elif SERVER_JOB_ID=$(sbatch --parsable --partition=coe-gpu --export=ALL server.sh); then
  export LLMGE_SERVER_JOB_ID=${SERVER_JOB_ID%%;*}
  printf '%s
' "$LLMGE_SERVER_JOB_ID" > run_job_outputs/dfine_current_server.id
  echo "Server job: $LLMGE_SERVER_JOB_ID"
  squeue -j "$LLMGE_SERVER_JOB_ID" -o '%i %T %R %L'
else
  echo 'Server submission failed. Do not start the controller.'
fi
```

After the new server reports readiness as in section 3:

```bash
if RUN_JOB_ID=$(sbatch --parsable --export=ALL run.sh); then
  RUN_JOB_ID=${RUN_JOB_ID%%;*}
  printf '%s
' "$RUN_JOB_ID" > run_job_outputs/dfine_current_controller.id
  echo "Resumed controller job: $RUN_JOB_ID"
  squeue -j "$RUN_JOB_ID" -o '%i %T %R %L'
else
  echo 'Controller submission failed.'
fi
```

## 8. Check generation-20 completion

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
export FRED_RUN_TAG="$(cat run_job_outputs/dfine_current_tag.txt)"
RUN_JOB_ID="$(cat run_job_outputs/dfine_current_controller.id)"
sacct -j "$RUN_JOB_ID" --format=JobID,State,ExitCode -P
.venv/bin/python -c 'import json,sys; s=json.load(open(sys.argv[1])); assert s.get("status") == "complete" and s.get("last_completed_generation") == 20, s; print(s)' "fred_runs/$FRED_RUN_TAG/status.json"
test -s "fred_runs/$FRED_RUN_TAG/checkpoints/checkpoint_gen_20.pkl" && echo 'Generation 20 checkpoint ready'
test -s "fred_runs/$FRED_RUN_TAG/checkpoints/global_gen_20.pkl" && echo 'Generation 20 global checkpoint ready'
```

## 9. Generate search result plots and tables

```bash
export FRED_SEED_FAMILY=dfine
export FRED_RUN_TAG="$(cat run_job_outputs/dfine_current_tag.txt)"
.venv/bin/python analysis/scripts/fred_pareto_graphs.py \
  --family "$FRED_SEED_FAMILY" \
  --checkpoints "fred_runs/$FRED_RUN_TAG/checkpoints" \
  --models_dir "fred_runs/$FRED_RUN_TAG/candidates/models" \
  --output "analysis/results/analysis/fred_pareto/$FRED_RUN_TAG"
test -s "analysis/results/analysis/fred_pareto/$FRED_RUN_TAG/pareto2d_final.png" && echo 'Pareto plot ready'
ls -lh "analysis/results/analysis/fred_pareto/$FRED_RUN_TAG"
```